# <span style="color:red">UNDER CONSTRUCTION!!!!</span>

# Spoken Language Processing - Instituto Superior Técnico
## Laboratory Assignment 2 - Native Language Identification challenge

# WEEK 2 - Using pre-trained models


During this week, students will implement two modern systems for native language identification based on:
- speaker representations (utterance-based) obtained with an x-vector model (`lab2_xvec.ipynb` notebook); 
- speech representations (frame-based) obtained with a self-supervised learning (SSL) pre-trained model (this notebook). 

In both cases, students are encouraged to explore different feature configurations and alternative downstream models.

## Before starting

Let's import some modules and make some definitions:

In [ ]:
import os 
import csv 
import pickle
import numpy as np
import librosa 
import torch 

from pf_tools import CheckThisCell, ETS
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt

LANGUAGES = ('ARA',  'KOR',  'SPA',  'TUR')
LANG2ID = {'ARA':0, 'KOR':1, 'SPA':2, 'TUR':3}
ID2LANG = dict((LANG2ID[k],k)for k in LANG2ID)

Like in the previous Notebooks, you need to mount Google drive if you are working on Google Colab. Otherwise, you should skip or delete the following code cell:

In [ ]:

raise CheckThisCell ## <---- Remove this torun this cell if you are on Google Colab
from google.colab import drive
drive.mount('/content/drive')


Like in week1, the audio data is expected to be in a folder with the following format:

```
ets_data/
├── train/
│   └── audio/
│       └──wav files
│   └── key.lst 
│
└── train100/
    └── audio/
        └──wav files
    └── key.lst
... 
```

You must already have this from the previous week, so you can set-up your data directory:

In [ ]:

raise CheckThisCell ## <---- Remove this after completing/checking this cell

CWD = os.getcwd()
DATADIR = f'{CWD}/ets_data/' # <--- Change this variable to your working directory containig the ETS data
if not os.path.isdir(DATADIR):
    os.mkdir(DATADIR)
    print(f'WARNING: Your data is not in the folder {DATADIR}')

os.chdir(CWD)
print(f'Your ETS data should be in this folder {DATADIR}')


If you need to download again the data, you can run the following cell:

In [ ]:
raise CheckThisCell

os.chdir(DATADIR)

# download train
!wget http://groups.tecnico.ulisboa.pt/speechproc/pf25/lab2/train.tgz
!tar -xzvf train.tgz

#download train100
!wget http://groups.tecnico.ulisboa.pt/speechproc/pf25/lab2/train100.tgz
!tar -xzvf train100.tgz

#download dev
!wget http://groups.tecnico.ulisboa.pt/speechproc/pf25/lab2/dev.tgz
!tar -xzvf dev.tgz

#download evl
!wget http://groups.tecnico.ulisboa.pt/speechproc/pf25/lab2/evl.tgz
!tar -xzvf evl.tgz

os.chdir(CWD)

## Using self-supervised pre-trained models (SSL)

The goal of this part of the laboratory is to expose students to modern tools and methods for speech classification.
In particular, we will explore self-supervised learning (SSL) models (available at [HuggingFace](https://huggingface.co/)) to build a native language identification system.

We will explore two approaches:

- A system that follows the same structure of the last x-vector  system: a pre-trained model (typically referred to as upstream model) to extract features followed by a decoupled downstream neural model. There will be no adaptation of the upstream model. This is sometimes refered in the literature as *linear probing*.

- A system that  consists of a classification head on top of the  SSL model, but in this case, the entire network will be fine-tuned (this process can be significantly slower than any previous system we trained so far).

Notice that when comparing SSL features with x-vector features there are two fundamental differences that impact our classifiers:
1. SSL models are pre-trained in a self-supervised way, thus, potentially with larger amounts of data;
2. SSL models produce features at the frame-level.

In this part of the lab, students are  expected to *play* with the different upstream models to build the best possible native language identification system. 

In particular, students are encouraged to explore and discover which of the available SSL models can be a better candidate for their classification system. Potential candidates are [HuBERT](https://huggingface.co/docs/transformers/model_doc/hubert), [wav2vec2](https://huggingface.co/docs/transformers/model_doc/wav2vec2), and [WavLM](https://huggingface.co/docs/transformers/model_doc/wavlm) among others. Note that using a large SSL model will make the feature extraction process quite slow and the fine-tuning **VEEEERY SLOW**. Notice also that there may be different versions available of each model, trained with different amounts of data or with incresed number of parameters.




## 1. SSL model as a (frozen) feature extractor

### 1.1 Extracting SSL features

The following code snipet shows how to load an SSL model from huggingface, load an audio file and preprocess it.

In [ ]:

from transformers import Wav2Vec2FeatureExtractor
from transformers import Wav2Vec2Model

# Load pretrained model and processor
# You can use any Wav2Vec2 model from Hugging Face Model Hub
# https://huggingface.co/models?search=wav2vec2
# For example, you can use "facebook/wav2vec2-base-960h" or "facebook/wav2vec2-large-960h"
model_name = "facebook/wav2vec2-base"

# Load the processor and model
processor = Wav2Vec2Processor.from_pretrained(model_name)
model = Wav2Vec2Model.from_pretrained(model_name)

# Load an example audio file
audiofile = f'{DATADIR}/train100/audio/train_0003.wav'
audio, _ = librosa.load(audiofile, sr=16000)
inputs = processor(audio, sampling_rate=16000, return_tensors="pt", padding=True)


Then, we will pass the processed input through the model. We will use as features the activations of the last hidden layer. Later, you can play with the specific layer used. Researchers have shown that some middle layers can contain more information for tasks similar to ours. 

In [ ]:
with torch.no_grad():
    outputs = model(**inputs, output_hidden_states=True)
    last_hidden_states = outputs.hidden_states[-1]  # Get the last hidden states
    print(inputs['input_values'].shape)  # (batch_size, sequence_length)
    print(last_hidden_states.shape)  # (batch_size, sequence_length, hidden_size)
    
   

Notice the dimension of the input and the output. What is the relation between them? What is the frame rate of the SSL model?

Use this example to write a function that takes as arguments an audio filename, an SSL preprocessor and an SSL model and returns a numpy array of dimension (1xD) (you can compute the mean to reduce the time dimension). 

In [ ]:
raise CheckThisCell ## <---- Remove this after completing/checking this cell

def extract_wav2vec(filename, processor, model, duration=10.0, num_layer=-1):
    # LABWORK :: CODE TO INSERT HERE
    pass
    # LABWORK :: CODE TO INSERT HERE

# This must return a numpy array of shape (1, 768)
feat = extract_wav2vec(f'{DATADIR}/train100/audio/train_0003.wav', processor, model)
print(feat.shape, type(feat))

The same way as we did with all the previous systems, we will define the feature extraction configuration using a dictionary:

In [ ]:
raise CheckThisCell ## <---- Remove this after completing/checking this cell

transform['wav2vec2'] = {
                    'audio_transform': 
                        lambda x : extract_wav2vec(x, 
                            processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base"),
                            model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")
                            ), ## <--- You need to modify this here
                    'chunk_transform': None,
                    'chunk_size': 0,
                    'chunk_hop':0   
                }

transform['wav2vec2_7thlayer'] = {
                    'audio_transform': 
                        lambda x : extract_wav2vec(x, 
                            processor = Wav2Vec2FeatureExtractor.from_pretrained("facebook/wav2vec2-base"),
                            model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base"), 
                            num_layer=7
                            ), ## <--- You need to modify this here
                    'chunk_transform': None,
                    'chunk_size': 0,
                    'chunk_hop':0   
                }

And instantiate the ETS class for all the data partitions to apply the feature extraction. **WARNING** This can be very slow depending on the model and the available computational resources.

In [ ]:

# Download and feature extract
trainset = 'train100'
transform_id = 'wav2vec2' 

ets_partitions = {}
# for partition in ('train', 'train100', 'dev', 'evl'):
for partition in ('train100', 'dev', 'evl'):
    ets_partitions[partition] = ETS(DATADIR, partition, 
                    transform_id=transform_id, 
                    audio_transform=transform[transform_id]['audio_transform'], 
                    chunk_transform=transform[transform_id]['chunk_transform'],
                    chunk_size=transform[transform_id]['chunk_size'], 
                    chunk_hop=transform[transform_id]['chunk_hop']
                    )


### 1.2 Training the upstream model (frozen)

As mentioned before, our first attempt of native language identification system with SSL features will be a simple neural model (4-classes) like in the last `x-vector` model. We could have used also the SVM or any other model. Students are encouraged to explore alternative neural model configurations.


In [ ]:
raise CheckThisCell ## <---- Remove this after completing/checking this cell
feat_dim = 0 ### <--- You need to modify this here

# Define a simple linear classification model
class LinearClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super(LinearClassifier, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), 
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, num_classes)  
        )
    
    def forward(self, x):
        return self.model(x)

model = LinearClassifier(input_dim=feat_dim, hidden_dim=200, num_classes=len(LANGUAGES))

In [ ]:
from pf_tools import train_nn, save_model

train_nn(model, ets_partitions[trainset], ets_partitions['dev'], class2id=LANG2ID, batch_size=16, epochs=5000)

model_id = save_model(model, f'nnet_{transform_id}', f'{DATADIR}/{trainset}/models/')
print(f'Model {model_id} saved in {DATADIR}/{trainset}/models/')

### 1.3 Analyze results on the dev set and prepare your submission file

Let's check our performance on the dev set:

In [ ]:
from pf_tools import plot_confusion_matrix 
from pf_tools import predict_nn

# Predict the dev set
hyp, ref, files = predict_nn(model, ets_partitions['dev'], class2id=LANG2ID)
filename = f'{DATADIR}/{trainset}/models/{model_id}/dev.pkl'
pickle.dump({'hyp':hyp, 'fileids':files}, open(filename, 'wb'))

# Report the results
print(classification_report(ref, hyp))
print(accuracy_score(ref, hyp))
cm = confusion_matrix(ref, hyp)
plot_confusion_matrix(cm, LANGUAGES, title=f'Confusion Matrix\n(Accuracy {100*accuracy_score(ref, hyp):.2f})')

# Predict the evl set
hyp, ref, files = predict_nn(model, ets_partitions['evl'])
filename = f'{DATADIR}/{trainset}/models/{model_id}/evl.pkl'
pickle.dump({'hyp':hyp, 'fileids':files}, open(filename, 'wb'))

If things went as expected, you should obtain around 66.1% accuracy with the 7th layer of the wav2vec2base.

At this point, you can explore different x-vector model configurations for feature extraction and alternative neural model architectures. You can also generate the final prediction file and make a submission to the  [Kaggle competition](https://www.kaggle.com/t/312cd4200cfb4e138ea9372ce5bc33fd):

In [ ]:
from pf_tools import create_submission_file

students_group = '00' # <--- CHANGE THIS ACCORDINGLY

# model_id = 'svm_spkrec-ecapa-voxceleb_2025-05-04_19:10:21'
model_id_short = 'nnet_wv2v2'

results_path = f'{DATADIR}/{trainset}/models/{model_id}/'
filename = f'{CWD}/g{students_group}_{trainset}_{model_id_short}.csv' # <--- CHANGE THIS ACCORDINGLY

create_submission_file(results_path, filename)

## 2. Fine-tuning the SSL model

The full potential of SSL models is  attained when we finetune the complete model for the target task. 

The following code snipet shows how to load an SSL model from huggingface, load an audio file and preprocess it.

# Contacts and support
You can contact the professors during the classes or the office hours.

Particularly, for this second laboratory assignment, you should contact Prof. Alberto Abad: alberto.abad@tecnico.ulisboa.pt


